#### Loading Model From Drive

In [1]:
import shutil
import torch
from torch import nn
from transformers import AutoModelForCausalLM
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
shutil.copy('/content/drive/MyDrive/AI Safety /Lab/Model/reward_model.pt', '/content')
shutil.copy('/content/drive/MyDrive/AI Safety /Lab/Model/sft_model.zip', '/content')
print("Model Copied to workspace")

Model Copied to workspace


In [3]:
# unzipping Model
!unzip -q sft_model.zip
print("Model Unzipped")

replace sft_model/model.safetensors? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y
replace sft_model/config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: replace sft_model/generation_config.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Model Unzipped


#### Reward Model Implementation

In [4]:
class RewardHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.hidden_size = config.hidden_size
    self.reward = nn.Linear(self.hidden_size, 1)
    self._post_init()


  def _post_init(self):
    nn.init.normal_(self.reward.weight, std=(1.0/np.sqrt(self.hidden_size + 1)))
    nn.init.zeros_(self.reward.bias)

  def forward(self, hidden_state):
    return self.reward(hidden_state)


class GPT2RewardHead(nn.Module):
  def __init__(self, model_name):
    super().__init__()
    self.llm = AutoModelForCausalLM.from_pretrained(model_name)
    self.reward_head = RewardHead(self.llm.config)

  def forward(self, input_ids, attention_mask):
    transformer_outputs = self.llm.forward(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True
    )
    last_hidden_state = transformer_outputs.hidden_states[-1]
    reward = self.reward_head(last_hidden_state).squeeze(-1)
    return torch.sigmoid(reward)



##### Loading Our Model

In [5]:
reward_model = GPT2RewardHead('gpt2')
reward_model.load_state_dict(torch.load('reward_model.pt', map_location='cpu'))


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<All keys matched successfully>

#### Model with Value Head

In [6]:
class ValueHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.hidden_size = config.hidden_size
    self.value = nn.Linear(self.hidden_size, 1)
    self._post_init()


  def _post_init(self):
    nn.init.normal_(self.value.weight, std=(1.0/np.sqrt(self.hidden_size + 1)))
    nn.init.zeros_(self.value.bias)

  def forward(self, hidden_state):
    output = hidden_state
    return self.value(output)


class ModelForCausualLLMWithValueHead(nn.Module):
  def __init__(self, model_path):
    super().__init__()
    self.llm = AutoModelForCausalLM.from_pretrained(model_path)
    self.v_head = ValueHead(self.llm.config)

  def forward(self, input_ids, attention_mask):
    transformer_outputs = self.llm.forward(
        input_ids=input_ids,
        attention_mask=attention_mask,
        output_hidden_states=True,
        return_dict=True
    )
    lm_logits = transformer_outputs.logits
    # Get the Last hidden state
    last_hidden_state = transformer_outputs.hidden_states[-1]

    # Applying the Value Head
    value = self.v_head(last_hidden_state).squeeze(-1)

    return lm_logits, value

  def generate(self, *args, **kwargs):
      return self.llm.generate(*args, **kwargs)



In [7]:
model_path = '/content/sft_model'
model = ModelForCausualLLMWithValueHead(model_path)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

###### Preparing Dataset


In [8]:
from transformers import AutoTokenizer
model_name = 'gpt2'

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [10]:
!pip -q install datasets

In [11]:
from datasets import load_dataset


In [12]:
dataset = load_dataset('sst2')
dataset

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [13]:
ds_train, ds_val = dataset['train'], dataset['validation']

###### Filtereing the Data

In [14]:


print(len(ds_train), len(ds_val))

67349 872


In [15]:
ds_train =  ds_train.filter(lambda x: len(x['sentence'].split(' ')) > 8)
ds_val =  ds_val.filter(lambda x: len(x['sentence'].split(' ')) > 8)

In [16]:
print(len(ds_train), len(ds_val))

31105 807


##### Applying Random split

In [17]:
import random

In [19]:
input_min_token_length = 2
input_max_token_length = 8
input_token_length_range = list(range(input_min_token_length, input_max_token_length))
print(input_token_length_range)

[2, 3, 4, 5, 6, 7]


In [22]:
random.choice(input_token_length_range)

2

In [23]:
def tokenize(sample):
  input_size = random.choice(input_token_length_range)
  sample['input_ids'] = tokenizer.encode(sample['sentence'])[:input_size]
  sample['attention_mask'] = [1] * len(sample['input_ids'])
  sample['query'] = tokenizer.decode(sample['input_ids'])
  return sample


map_kwargs = {
    "batched":False,
    "remove_columns":['idx', 'sentence', 'label']
}

# Applying to our Dataset
tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

Map:   0%|          | 0/31105 [00:00<?, ? examples/s]

Map:   0%|          | 0/807 [00:00<?, ? examples/s]

#### Converting to Tensors

In [24]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')


In [25]:
tokenized_dataset_train[5]

{'input_ids': tensor([ 533,  517, 7744, 1807,  832,  621,  287]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1]),
 'query': 'are more deeply thought through than in'}

In [26]:
# Defining our Reward Token ID that will be attarch to the model
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [28]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [29]:
batch_size = 32
def collator(batch):
  return dict((key, [d[key] for d in batch]) for key in batch[0])

train_dataloader = DataLoader(tokenized_dataset_train, shuffle=True, batch_size=batch_size, collate_fn=collator)
val_dataloader = DataLoader(tokenized_dataset_val, shuffle=False, batch_size=batch_size, collate_fn=collator)

In [30]:
batch = next(iter(train_dataloader))
batch

{'input_ids': [tensor([24089,  1254,   616, 29708,  2340]),
  tensor([27594,   870,   281,   267, 13034,    12]),
  tensor([20123,   274]),
  tensor([28004,   605,  2355,  2634,   837,   351,  7026]),
  tensor([ 83, 441, 829, 262]),
  tensor([ 1712,  2709, 14209]),
  tensor([   69, 25329,  1497]),
  tensor([ 270,  705,   82,  655, 1327]),
  tensor([   64,  1263,    12, 37315,    14,   439,    12]),
  tensor([9776, 1498]),
  tensor([  272, 42241]),
  tensor([14150,   587,  1365,   572, 10589,   319]),
  tensor([ 1169, 49291,   287, 46038,   318,   884]),
  tensor([ 1169, 22527,  1067,   463,   929,   468]),
  tensor([  272,  8323,  1146, 13443,   837]),
  tensor([ 6381,  1681, 42778,   262]),
  tensor([  292,   351,   477, 14742]),
  tensor([ 271,  257, 3348, 6131]),
  tensor([265, 663]),
  tensor([5562, 4952, 3923,  326,  670, 1377,  318]),
  tensor([1169, 9867]),
  tensor([2946,   82, 1393]),
  tensor([26024,   503,  2644]),
  tensor([ 293,  364,  837, 6011]),
  tensor([   64, 24769, 

#### Sampling the Response

In [31]:
output_min_length = 5
output_max_length = 16

generation_kwargs ={
    "min_length":-1,
    "top_k":0.0,
    "top_p":1.0,
    "do_sample":True,
    "pad_token_id":tokenizer.eos_token_id
}

##### Definig Simple Generation

In [35]:
new_token = random.choice(list(range(output_min_length, output_max_length)))
generation_kwargs["max_new_tokens"] = new_token
sample = tokenizer('Hi, this')
sample

{'input_ids': [17250, 11, 428], 'attention_mask': [1, 1, 1]}

In [37]:
query_response = model.generate(
    input_ids=torch.tensor(sample['input_ids']).unsqueeze(0),
    attention_mask=torch.tensor(sample['attention_mask']).unsqueeze(0),
    **generation_kwargs
)

query_response

tensor([[17250,    11,   428, 32327,  1641,  2646,   220,   220,   364, 27906,
         19907,  1842, 23411,   318,  8941,   257,   649,  2126]])

In [38]:
tokenizer.decode(query_response)

['Hi, this delightful family film  ersatz indie lovefest is hardly a new idea']

In [42]:
with torch.no_grad():
  query_response_score = torch.cat([query_response, torch.tensor([[REWARD_TOKEN_ID]])], dim=1)
  attention_mask = torch.ones_like(query_response_score, dtype = torch.long)
  score = reward_model(query_response_score, attention_mask).squeeze(0)[-1]
score

tensor(0.7087)